## NOTEBOOK- Análise Exploratória do Dataset de Inadimplência


### Importação das bibliotecas


In [1]:
import sys
sys.path.append("..")  # permite importar o pacote src/ a partir de notebooks/

import matplotlib.pyplot as plt

from src.data import carregar_dataset
from src.config import COLUNAS_FATURAS, COLUNAS_PAGAMENTOS_ANTERIORES, COLUNAS_HISTORICO_PAGAMENTO


### Carregamento de dataset


In [2]:
# Na primeira execução, baixa da UCI e salva em data/raw/.
# Nas próximas, lê direto do disco (não depende mais da API/da internet).
X, y = carregar_dataset()

print("Features:")
display(X.head())
print("\nTarget:")
display(y.head())


Features:


,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,...,X14,X15,X16,X17,X18,X19,X20,X21,X22,X23
0,20000,2,2,1,24,2,2,-1,-1,-2,...,689,0,0,0,0,689,0,0,0,0
1,120000,2,2,2,26,-1,2,0,0,0,...,2682,3272,3455,3261,0,1000,1000,1000,0,2000
2,90000,2,2,2,34,0,0,0,0,0,...,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000
3,50000,2,2,1,37,0,0,0,0,0,...,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000
4,50000,1,2,1,57,-1,0,-1,0,0,...,35835,20940,19146,19131,2000,36681,10000,9000,689,679



Target:


0    1
1    1
2    0
3    0
4    0
Name: Y, dtype: int64

### dimensão dados 


In [ ]:
print("Dimensão das features:", X.shape)
print("Dimensão do target:", y.shape)


### identificação das colunas e tipos


In [ ]:
print("Colunas das features:")
print(X.columns.tolist())

print("Tipos das features:")
print(X.dtypes)

print("\nTipo do target:")
print(y.dtypes)


### informação geral


In [ ]:
X.info()

y.info()


### valores ausentes


In [ ]:
print("Valores ausentes por coluna:")
display(X.isnull().sum())
print("Total de valores ausentes:", X.isnull().sum().sum())
print("Valores ausentes no target:", y.isnull().sum().sum())


### registros duplicados


In [ ]:
print("Duplicatas em X:", X.duplicated().sum())

dados = X.copy()
dados["Y"] = y.values
print("Duplicatas completas:", dados.duplicated().sum())


### duplicações nas features


In [ ]:
duplicados_X = dados[
    dados.duplicated(subset=X.columns, keep=False)
]

print(
    "Registros envolvidos em duplicações de X:",
    len(duplicados_X)
)

display(
    duplicados_X
    .sort_values(by=X.columns.tolist())
    .head(20)
)


### X iguais, mas Y diferentes


In [ ]:
grupos = (
    dados
    .groupby(X.columns.tolist())["Y"]
    .nunique()
)

print(
    "Grupos de X idênticos com Y diferente:",
    (grupos > 1).sum()
)


### estatisticas descritivas


In [ ]:
display(X.describe().T)


### variáveis categóricas


In [ ]:
#X2 → sexo
#X3 → educação
#X4 → estado civil

for coluna in ["X2", "X3", "X4"]:
    print(f"\n{coluna}")
    display(
        X[coluna]
        .value_counts()
        .sort_index()
    )


### histórico de pagamento


In [ ]:
for coluna in COLUNAS_HISTORICO_PAGAMENTO:
    print(f"\n{coluna}")
    display(
        X[coluna]
        .value_counts()
        .sort_index()
    )


### Variáveis financeiras



In [ ]:
# Listas de colunas agora centralizadas em src/config.py, evitando
# redefinição em cada notebook.
display(X[COLUNAS_FATURAS].describe().T)
display(X[COLUNAS_PAGAMENTOS_ANTERIORES].describe().T)


### valores negativos nas faturas


In [ ]:
for coluna in COLUNAS_FATURAS:
    negativos = (X[coluna] < 0).sum()

    print(
        f"{coluna}: {negativos} valores negativos"
    )


for coluna in COLUNAS_FATURAS:

    negativos = X.loc[
        X[coluna] < 0,
        coluna
    ]

    print(f"\n{coluna}")
    print(f"Quantidade: {len(negativos)}")
    print(f"Mínimo: {negativos.min()}")
    print(f"Mediana: {negativos.median()}")
    print(f"Máximo: {negativos.max()}")


### clientes com fatura negativa


In [ ]:
tem_negativo = (
    X[COLUNAS_FATURAS] < 0
).any(axis=1)

print(
    "Clientes com pelo menos uma fatura negativa:",
    tem_negativo.sum()
)

print(
    "Percentual:",
    tem_negativo.mean() * 100
)


quantidade_negativos = (
    X[COLUNAS_FATURAS] < 0
).sum(axis=1)

display(
    quantidade_negativos
    .value_counts()
    .sort_index()
)


### valores extremos em faturas


In [ ]:
for coluna in COLUNAS_FATURAS:

    print(f"\n{coluna} - menores valores:")
    display(X[coluna].nsmallest(10))

    print(f"\n{coluna} - maiores valores:")
    display(X[coluna].nlargest(10))


### Variável alvo


In [ ]:
print("Distribuição do target:")
display(y.value_counts())

print("Proporção do target:")
display(y.value_counts(normalize=True) * 100)


### visualização das variáveis


#### Variável alvo


In [ ]:
y.value_counts().plot(kind="bar")

plt.title("Distribuição da inadimplência")
plt.xlabel("Default")
plt.ylabel("Quantidade")

plt.show()


#### variáveis numéricas


In [ ]:
plt.figure(figsize=(8, 5))
X["X1"].hist(bins=30)
plt.title("Distribuição do limite de crédito")
plt.xlabel("Limite de crédito")
plt.ylabel("Quantidade")
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
X["X5"].hist(bins=30)
plt.title("Distribuição da idade")
plt.xlabel("Idade")
plt.ylabel("Quantidade")
plt.show()


#### Variáveis financeiras


In [ ]:
plt.figure(figsize=(10, 6))
X[COLUNAS_FATURAS].boxplot()
plt.title("Distribuição dos valores das faturas")
plt.ylabel("Valor")
plt.show()


plt.figure(figsize=(10, 6))
X[COLUNAS_PAGAMENTOS_ANTERIORES].boxplot()
plt.title("Distribuição dos pagamentos anteriores")
plt.ylabel("Valor")
plt.show()


### Correlção


In [ ]:
correlacao = X.corr()
display(correlacao)

plt.figure(figsize=(12, 10))
plt.imshow(correlacao, aspect="auto")
plt.colorbar()
plt.xticks(range(len(correlacao.columns)), correlacao.columns, rotation=90)
plt.yticks(range(len(correlacao.columns)), correlacao.columns)
plt.title("Matriz de correlação")
plt.show()


##  Análise Exploratória de Dados (EDA) — Previsão de Inadimplência

---

### Conclusão da Análise Exploratória

A análise exploratória permitiu compreender a estrutura, a qualidade e as principais características do conjunto de dados utilizado para a previsão de inadimplência. O dataset possui **30.000 registros** e **23 variáveis preditoras**, além da **variável alvo Y**, que representa o pagamento ou inadimplência no mês seguinte. Não foram identificados valores ausentes nas variáveis analisadas.

####  Qualidade e Integridade dos Dados

* **Registros Duplicados:** Durante a análise, foram identificados **56 registros** com características duplicadas:
  * **35 registros completamente duplicados** (considerando inclusive a variável alvo $Y$). Estes serão removidos na etapa de preparação dos dados.
  * **21 grupos de registros** que possuem exatamente as mesmas características de entrada ($X$), porém com valores opostos para a variável alvo ($Y$). Como não é possível afirmar que representam erros, esses casos serão mantidos no dataset.
* **Variáveis Categóricas ($X3$ e $X4$):** Nas variáveis $X3$ (educação) e $X4$ (estado civil), foram encontrados códigos que não são explicados pela documentação consultada. Como não é seguro determinar seus significados individuais, esses códigos serão agrupados na categoria referente a "outros", evitando interpretações arbitrárias.

####  Variáveis Financeiras e Histórico de Pagamentos

* **Histórico de Pagamentos ($X6$ a $X11$):** Apresentam valores no intervalo entre $-2$ e $8$. Por aparecerem com frequência significativa e fazerem parte da estrutura original do conjunto de dados, serão preservados no primeiro experimento de modelagem, sem conversão para valores ausentes.
* **Valores de Fatura ($X12$ a $X17$):** Foram identificados valores negativos sem explicação prévia na documentação oficial. Decidiu-se mantê-los no primeiro modelo para posteriormente avaliar seu real impacto no desempenho do algoritmo.
* **Escala e Valores Extremos:** A análise das estatísticas descritivas apontou grande diferença de escala e a presença de valores extremos (*outliers*) nas variáveis financeiras, especialmente nos valores das faturas e pagamentos anteriores. Essa característica será diretamente tratada nas etapas de pré-processamento e seleção dos algoritmos.

---

####  Resumo da Análise

Com base nos resultados da EDA, o conjunto de dados apresenta condições adequadas para avançar para a etapa de pré-processamento e construção dos modelos de Machine Learning. As decisões tomadas durante essa etapa serão mantidas de forma explícita para que os resultados dos modelos possam ser reproduzidos e avaliados posteriormente. Essas decisões estão implementadas em `src/preprocessing.py`, para que o mesmo tratamento seja aplicado de forma consistente em todos os notebooks.

---

###  Decisões para o Pré-processamento

| Problema encontrado | Decisão |
| :--- | :--- |
| **Valores ausentes** | Nenhum tratamento necessário |
| **35 duplicatas completas** | Remover |
| **21 grupos com X igual e Y diferente** | Manter |
| **X3 com códigos não documentados** | Agrupar como "Outros" |
| **X4 com código não documentado** | Agrupar como "Outros" |
| **X6–X11 com códigos -2 a 8** | Preservar inicialmente |
| **Valores negativos em X12–X17** | Preservar inicialmente |
| **Escalas diferentes** | Tratar no pré-processamento |
| **Valores extremos** | Investigar, sem remoção automática |
